In [14]:
import os
from enum import StrEnum

import polars as pl
import pandas as pd
import gseapy as gp
import synapseclient
from synapseclient.entity import File as sceFile 

In [66]:
# Jupyter server should be running from src/python
LIMMA_RESULTS = os.path.join(
    os.path.dirname(os.path.dirname(os.getcwd())),
    "analysis",
    "mutations",
    "_cache",
    "limma",
    "limma_results.parquet"
)

GSEA_DIR = os.path.join(
    os.path.dirname(os.path.dirname(os.getcwd())),
    "analysis",
    "mutations",
    "_cache",
    "gsea",
)
GSEA_RESULTS = os.path.join(
    GSEA_DIR,
    "gsea_results.parquet"
)

GSEA_RESULTS_SYNID = "syn78432275"

In [30]:
class GENE_SETS(StrEnum):
    HALLMARK = "MSigDB_Hallmark_2020"
    KEGG = "KEGG_2021_Human"
    REACTOME = "Reactome_2022"

In [31]:
# filter to gene-centric omics layers
# (transcriptomics and global proteomics)
tx_px_results = (
    pl.read_parquet(LIMMA_RESULTS)
    .filter(
        pl.col("OmicsType")
        .is_in(["Transcriptomics", "Proteomics"])
    )
)

In [33]:
gsea_results_list = []

for (omics_type, contrast), df_sub in tx_px_results.group_by(["OmicsType", "Contrast"]):
    print(f"Running GSEA for {omics_type} | {contrast}...")
    
    # Extract feature ranking metric: Feature (Gene symbol) and limma's t-statistic
    ranked_series = (
        df_sub.select(["Feature", "t"])
        .filter(pl.col("Feature").is_not_null() & pl.col("t").is_not_null())
        # Deduplicate features if any exist, keeping highest absolute t-stat
        .with_columns(abs_t=pl.col("t").abs())
        .sort("abs_t", descending=True)
        .unique(subset=["Feature"], keep="first")
        .sort("t", descending=True)
        .to_pandas()
        .set_index("Feature")["t"]
    )
    
    # Run Preranked GSEA
    for gene_set in [
        GENE_SETS.HALLMARK,
        GENE_SETS.KEGG,
        GENE_SETS.REACTOME
    ]:
        try:
            prerank = gp.prerank(
                rnk=ranked_series,
                gene_sets=gene_set,
                threads=8,
                min_size=15,
                max_size=500,
                permutation_num=1000,
                seed=42069,
                verbose=False
            )
            
            # Format result
            out_df = prerank.res2d.copy()
            out_df["OmicsType"] = omics_type
            out_df["Contrast"] = contrast
            out_df["GeneSet"] = gene_set
            gsea_results_list.append(out_df)
        except Exception as e:
            print(f"Failed for {omics_type} {contrast}: {e}")

# Combine into a single tidy dataframe
all_gsea = pl.from_pandas(pd.concat(gsea_results_list, ignore_index=True))
all_gsea.write_parquet(GSEA_RESULTS)

Running GSEA for Proteomics | Black_NPM1+/FLT3-ITD+_vs_White_NPM1+/FLT3-ITD+...
Running GSEA for Transcriptomics | Black_NPM1+/FLT3-ITD-_vs_Black_NPM1-/FLT3-ITD-...
Running GSEA for Proteomics | Black_NPM1+/FLT3-ITD-_vs_White_NPM1+/FLT3-ITD-...
Running GSEA for Transcriptomics | Black_NPM1-/FLT3-ITD+_vs_Black_NPM1-/FLT3-ITD-...
Running GSEA for Transcriptomics | Black_NPM1+/FLT3-ITD+_vs_White_NPM1+/FLT3-ITD+...
Running GSEA for Proteomics | White_NPM1+/FLT3-ITD+_vs_White_NPM1+/FLT3-ITD-...
Running GSEA for Transcriptomics | White_NPM1+/FLT3-ITD-_vs_White_NPM1-/FLT3-ITD-...
Running GSEA for Proteomics | Black_NPM1-/FLT3-ITD+_vs_Black_NPM1-/FLT3-ITD-...
Running GSEA for Proteomics | Black_NPM1-/FLT3-ITD-_vs_White_NPM1-/FLT3-ITD-...
Running GSEA for Transcriptomics | Black_NPM1+/FLT3-ITD+_vs_Black_NPM1+/FLT3-ITD-...
Running GSEA for Transcriptomics | White_NPM1+/FLT3-ITD+_vs_White_NPM1+/FLT3-ITD-...
Running GSEA for Proteomics | Black_NPM1+/FLT3-ITD+_vs_Black_NPM1-/FLT3-ITD+...
Running GS

In [45]:
# post-process GSEA results into a cleaner, more interpretable format
all_gsea = (
    all_gsea
    .filter(
        # strong enrichment scores
        (pl.col("NES").abs() >= 1.5)
        # conservative FDR, 0.1 is sometimes acceptable for discovery purposes
        & (pl.col("FDR q-val") <= 0.05)
    )
    .with_columns(
        (
            pl.col("Tag %")
            .str.split("/")
            .list.get(0)
            .cast(pl.Int64)
            .alias("leading_edge_size")
        ),
        (
            pl.col("Tag %")
            .str.split("/")
            .list.get(1)
            .cast(pl.Int64)
            .alias("gene_set_size")
        )
    )
    .with_columns(
        (100. * pl.col("leading_edge_size") / pl.col("gene_set_size"))
        .alias("Tag %")
    )
    .with_columns(
        pl.col("Gene %")
        .str.strip_suffix("%")
        .cast(pl.Float64)
    )
    .filter(
        # peak signal concentrated near top or bottom of ranked list
        # (more differential features and stronger signal)
        (pl.col("Gene %") <= 25)
        # a significant portion of pathway members were found
        & (pl.col("Tag %") >= 25)
    )
    # reorder columns (omit the ones we do not need)
    .select(
        "Contrast", "OmicsType", "GeneSet", "Term", "NES", "FDR q-val", 
        "Gene %", "Tag %", "leading_edge_size", "gene_set_size", "Lead_genes"
    )
)

all_gsea

Contrast,OmicsType,GeneSet,Term,NES,FDR q-val,Gene %,Tag %,leading_edge_size,gene_set_size,Lead_genes
str,str,str,str,f64,f64,f64,f64,i64,i64,str
"""Black_NPM1+/FLT3-ITD+_vs_White…","""Proteomics""","""MSigDB_Hallmark_2020""","""Myc Targets V1""",2.054229,0.0,23.99,57.731959,112,194,"""CDC45;PA2G4;RPS2;SRPK1;UBE2E1;…"
"""Black_NPM1+/FLT3-ITD+_vs_White…","""Proteomics""","""KEGG_2021_Human""","""Staphylococcus aureus infectio…",-3.573441,0.0,4.72,44.444444,20,45,"""FPR1;KRT9;KRT10;HLA-DMB;HLA-DM…"
"""Black_NPM1+/FLT3-ITD+_vs_White…","""Proteomics""","""KEGG_2021_Human""","""Graft-versus-host disease""",-2.616281,0.0,15.57,85.714286,18,21,"""HLA-DMB;HLA-DMA;HLA-G;HLA-DQA1…"
"""Black_NPM1+/FLT3-ITD+_vs_White…","""Proteomics""","""KEGG_2021_Human""","""Asthma""",-1.911691,0.003322,15.0,81.25,13,16,"""HLA-DMB;HLA-DMA;HLA-DQA1;HLA-D…"
"""Black_NPM1+/FLT3-ITD+_vs_White…","""Proteomics""","""KEGG_2021_Human""","""Dilated cardiomyopathy""",1.868212,0.023344,19.08,55.555556,15,27,"""ITGA2;ITGA2B;TGFB1;ITGB3;ATP2A…"
…,…,…,…,…,…,…,…,…,…,…
"""Black_NPM1+/FLT3-ITD-_vs_Black…","""Proteomics""","""Reactome_2022""","""Sema4D In Semaphorin Signaling…",-1.62075,0.03726,17.78,50.0,9,18,"""MYH11;MYH10;MYH14;RRAS;MYL6;AR…"
"""Black_NPM1+/FLT3-ITD-_vs_Black…","""Proteomics""","""Reactome_2022""","""Fanconi Anemia Pathway R-HSA-6…",-1.619351,0.037489,18.84,52.941176,9,17,"""USP1;CENPX;EME1;UBE2T;FANCD2;F…"
"""Black_NPM1+/FLT3-ITD-_vs_Black…","""Proteomics""","""Reactome_2022""","""RND2 GTPase Cycle R-HSA-969627…",-1.618669,0.037307,23.09,40.909091,9,22,"""PLXND1;CKAP4;ARHGAP5;GOLGA3;TF…"


In [58]:
with pl.Config(set_tbl_rows=20, set_fmt_str_lengths=80):
    display(all_gsea.group_by("Contrast").len().sort("Contrast"))

Contrast,len
str,u32
"""Black_NPM1+/FLT3-ITD+_vs_Black_NPM1+/FLT3-ITD-""",51
"""Black_NPM1+/FLT3-ITD+_vs_Black_NPM1-/FLT3-ITD+""",111
"""Black_NPM1+/FLT3-ITD+_vs_White_NPM1+/FLT3-ITD+""",23
"""Black_NPM1+/FLT3-ITD-_vs_Black_NPM1-/FLT3-ITD-""",175
"""Black_NPM1+/FLT3-ITD-_vs_White_NPM1+/FLT3-ITD-""",59
"""Black_NPM1-/FLT3-ITD+_vs_Black_NPM1-/FLT3-ITD-""",507
"""Black_NPM1-/FLT3-ITD+_vs_White_NPM1-/FLT3-ITD+""",276
"""Black_NPM1-/FLT3-ITD-_vs_White_NPM1-/FLT3-ITD-""",153
"""White_NPM1+/FLT3-ITD+_vs_White_NPM1+/FLT3-ITD-""",198


In [63]:
all_gsea.write_parquet(GSEA_RESULTS)

In [64]:
# Jupyter server should be running from src/python
# assumes auth_token.txt exists in src/python as described in setup instructions
AUTH_TOKEN = os.path.abspath("auth_token.txt")

with open(AUTH_TOKEN, "r") as atf:
    SYN = synapseclient.login(authToken=atf.read())


UPGRADE AVAILABLE

A more recent version of the Synapse Client (4.14.0) is available. Your version (4.11.0) can be upgraded by typing:
   pip install --upgrade synapseclient

Python Synapse Client version 4.14.0 release notes

https://python-docs.synapse.org/news/


Welcome, dylan.ross!



In [ ]:
REUPLOAD_GSEA_RESULTS = False

if REUPLOAD_GSEA_RESULTS:
    # upload limma results to synapse
    SYN.store(sceFile(path=GSEA_RESULTS, id=GSEA_RESULTS_SYNID))

/var/folders/nb/ykfs58f552d0d0jd_2jnkx_80000gp/T/ipykernel_14755/4269971038.py:5: DeprecationWarning: Call to deprecated method store. (To be removed in 5.0.0. Use `from synapseclient.operations import store` instead.) -- Deprecated since version 4.11.0.
  SYN.store(sceFile(path=GSEA_RESULTS, id=GSEA_RESULTS_SYNID))
